# TDN — Complete Faithful Implementation on KTH (v3)

This is the final, complete version incorporating all four source files:

| File | What it adds |
|---|---|
| `tdn_net.py` | `TDN_Net` architecture |
| `base_module.py` | `FBResNet`, `BottleneckShift`, `mSEModule`, `ShiftModule` |
| `transforms.py` | `GroupMultiScaleCrop`, `GroupRandomHorizontalFlip`, `GroupNormalize`, `Stack` — group-consistent spatial augmentation |
| `lr_scheduler.py` | `GradualWarmupScheduler` + `get_scheduler` — warmup + cosine/step LR schedule |

### What changed from v2

**Group transforms (correctness fix):**  
v2 applied `torchvision` transforms *independently* to each frame in a 5-frame window.  
This meant `RandomCrop` could pick a *different crop region* for each frame, making  
the temporal differences in `TDN_Net` meaningless (the network would see spatial  
offsets rather than motion). The group transforms apply **one shared random crop  
and flip decision** to all 5 frames in a segment.

**LR scheduler (training quality):**  
v2 used plain Adam at a fixed LR with a hard unfreeze. The original paper uses  
SGD + linear warmup + cosine (or step) decay. This notebook adds the full  
`GradualWarmupScheduler` and allows switching between `cosine` and `step` schedules.  
Adam is kept as default (works well for fine-tuning), but SGD + warmup is also provided.

### Architecture reminder
```
Per segment — 5 frames → [x1…x5]
  Diff stream:  diffs → conv1_5 → MaxPool → diff_layer1
  RGB stream:   x3   → conv1   → MaxPool
  Fusion 1:  α·RGB + β·diff_stem   (before layer1)
  Fusion 2:  α·RGB + β·diff_l1     (after  layer1)
  Layers 2–4: BottleneckShift (mSEModule + ShiftModule inside each block)
  Consensus:  mean over num_segments → logits
```


## Imports

In [ ]:
import os, re, random, json, math
from datetime import datetime

import cv2
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as TF
from torchvision.datasets.folder import make_dataset
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR, MultiStepLR, _LRScheduler
import kagglehub

import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
from torchmetrics.classification import MulticlassAccuracy, MulticlassConfusionMatrix
from sklearn.metrics import classification_report

## Seed

In [ ]:
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Seed set to {seed}")

## Group Transforms (from `transforms.py`)

These operate on a **list of PIL Images** — applying the *same* random decision  
(crop origin, flip coin) to every frame in the group. This is essential for TDN  
because the temporal differences computed inside `TDN_Net` must reflect genuine  
motion, not spatial jitter from inconsistent crops.

### Pipeline comparison

| Stage | v2 (per-frame torchvision) | v3 (group transforms) |
|---|---|---|
| Resize | `T.Resize(256)` per frame | `GroupScale(256)` — same for all |
| Crop | `T.RandomCrop(224)` — *independent* random origin per frame | `GroupMultiScaleCrop` — **one** origin for all 5 frames |
| Flip | `T.RandomHorizontalFlip()` — *independent* coin per frame | `GroupRandomHorizontalFlip` — **one** coin for all 5 frames |
| Stack | `torch.cat(frames)` after transforms | `Stack()` after group transform |
| Normalise | `T.Normalize()` per frame | `GroupNormalize` — repeated mean/std over 15 channels |

### `GroupMultiScaleCrop`
Randomly selects from 4 scales `[1, 0.875, 0.75, 0.66]` and up to 13 fixed  
crop positions (corners, edges, centre). Much richer than a plain `RandomCrop`.


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Group transforms — exact port of ops/transforms.py
# All transforms accept a list of PIL Images and return a list of PIL Images
# (or a stacked tensor for Stack).
# ════════════════════════════════════════════════════════════════════

class GroupRandomCrop:
    """Same random crop applied to every image in the group."""
    def __init__(self, size):
        self.size = (int(size), int(size)) if isinstance(size, (int, float)) else size
    def __call__(self, img_group):
        w, h   = img_group[0].size
        th, tw = self.size
        x1 = random.randint(0, w - tw)
        y1 = random.randint(0, h - th)
        return [img if (w==tw and h==th) else img.crop((x1, y1, x1+tw, y1+th))
                for img in img_group]

class GroupCenterCrop:
    """Centre crop applied to every image in the group."""
    def __init__(self, size):
        self.worker = TF.CenterCrop(size)
    def __call__(self, img_group):
        return [self.worker(img) for img in img_group]

class GroupScale:
    """Resize shorter edge to `size`, preserving aspect ratio."""
    def __init__(self, size):
        self.worker = TF.Resize(size)
    def __call__(self, img_group):
        return [self.worker(img) for img in img_group]

class GroupRandomHorizontalFlip:
    """One coin flip — all frames flipped or none."""
    def __call__(self, img_group):
        if random.random() < 0.5:
            return [img.transpose(Image.FLIP_LEFT_RIGHT) for img in img_group]
        return img_group

class GroupMultiScaleCrop:
    """
    Multi-scale crop with fixed anchor positions.

    Randomly samples one of 4 scales, then chooses from up to 13 fixed
    crop positions (corners, edges, centre, quarter-points).
    All frames in the group receive the SAME crop window.
    """
    def __init__(self, input_size, scales=None, max_distort=1,
                 fix_crop=True, more_fix_crop=True):
        self.scales        = scales or [1, .875, .75, .66]
        self.max_distort   = max_distort
        self.fix_crop      = fix_crop
        self.more_fix_crop = more_fix_crop
        self.input_size    = [input_size]*2 if isinstance(input_size, int) else input_size
        self.interpolation = Image.BILINEAR

    def __call__(self, img_group):
        cw, ch, ow, oh = self._sample_crop_size(img_group[0].size)
        cropped = [img.crop((ow, oh, ow+cw, oh+ch)) for img in img_group]
        return [img.resize(tuple(self.input_size), self.interpolation) for img in cropped]

    def _sample_crop_size(self, im_size):
        iw, ih   = im_size
        base     = min(iw, ih)
        crop_sizes = [int(base * s) for s in self.scales]
        ch = [self.input_size[1] if abs(x - self.input_size[1]) < 3 else x for x in crop_sizes]
        cw = [self.input_size[0] if abs(x - self.input_size[0]) < 3 else x for x in crop_sizes]
        pairs = [(w, h) for i, h in enumerate(ch)
                        for j, w in enumerate(cw) if abs(i-j) <= self.max_distort]
        cp = random.choice(pairs)
        if not self.fix_crop:
            return cp[0], cp[1], random.randint(0, iw-cp[0]), random.randint(0, ih-cp[1])
        offsets = self._fill_fix_offset(iw, ih, cp[0], cp[1])
        ow, oh  = random.choice(offsets)
        return cp[0], cp[1], ow, oh

    @staticmethod
    def _fill_fix_offset(iw, ih, cw, ch):
        ws = (iw - cw) // 4; hs = (ih - ch) // 4
        ret = [(0,0),(4*ws,0),(0,4*hs),(4*ws,4*hs),(2*ws,2*hs)]
        # 13-point grid (more_fix_crop=True)
        ret += [(0,2*hs),(4*ws,2*hs),(2*ws,4*hs),(2*ws,0),
                (ws,hs),(3*ws,hs),(ws,3*hs),(3*ws,3*hs)]
        return ret

class GroupNormalize:
    """
    Normalise a (N*3, H, W) tensor in-place.
    mean/std are repeated to match the number of channels (N*3).
    """
    def __init__(self, mean, std):
        self.mean = mean; self.std = std
    def __call__(self, tensor):
        # tensor: (N*3, H, W)
        rep_mean = self.mean * (tensor.size(0) // len(self.mean))
        rep_std  = self.std  * (tensor.size(0) // len(self.std))
        for t, m, s in zip(tensor, rep_mean, rep_std):
            t.sub_(m).div_(s)
        return tensor

class Stack:
    """
    Converts a list of N PIL RGB Images to a (N*3, H, W) float tensor.
    Each PIL image is converted via ToTensor (→ (3,H,W) in [0,1]),
    then all N tensors are concatenated along the channel dim.
    """
    def __call__(self, img_group):
        return torch.cat([TF.ToTensor()(img) for img in img_group], dim=0)  # (N*3, H, W)


# ── Convenience pipeline builders (used in Main Execution) ──────────────────
def make_train_transform():
    """
    Training pipeline:
      GroupMultiScaleCrop (4 scales, 13 anchor positions)
      GroupRandomHorizontalFlip
      Stack (PIL list → tensor)
      GroupNormalize (ImageNet stats, repeated across 15 channels)
    """
    return [
        GroupMultiScaleCrop(224, scales=[1, .875, .75, .66]),
        GroupRandomHorizontalFlip(),
        Stack(),
        GroupNormalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]

def make_eval_transform():
    """
    Eval/test pipeline:
      GroupScale(256) → GroupCenterCrop(224) → Stack → GroupNormalize
    """
    return [
        GroupScale(256),
        GroupCenterCrop(224),
        Stack(),
        GroupNormalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]

def apply_group_transform(frames, transform_list):
    """Apply a list of group transforms sequentially.
    frames: list of PIL Images
    Returns: (N*3, H, W) normalised float tensor
    """
    x = frames
    for t in transform_list:
        x = t(x)
    return x   # final output of Stack+GroupNormalize is a tensor

## LR Scheduler (from `lr_scheduler.py`)

### `GradualWarmupScheduler`
Linearly ramps the LR from `base_lr / multiplier` up to `base_lr` over  
`warmup_epoch × n_iter_per_epoch` steps, then hands off to a downstream scheduler.

### `get_scheduler`
Builds the full schedule in one call:

| `lr_scheduler` | Result |
|---|---|
| `'cosine'` | Warmup → CosineAnnealingLR down to `eta_min=1e-5` |
| `'step'`   | Warmup → MultiStepLR at `lr_steps` epochs |

### Why this matters vs plain Adam
The original paper trains with SGD + warmup + cosine and reports best results with  
the full schedule. For fine-tuning on a small dataset like KTH, Adam with warmup  
converges faster; the scheduler is included for both options.


In [ ]:
# ════════════════════════════════════════════════════════════════════
# LR Scheduler — exact port of ops/lr_scheduler.py
# ════════════════════════════════════════════════════════════════════

class GradualWarmupScheduler(_LRScheduler):
    """
    Linearly warm up LR from base_lr/multiplier → base_lr over warmup_epoch steps,
    then delegate to after_scheduler.

    Args:
        optimizer:        wrapped optimizer
        multiplier:       LR starts at base_lr / multiplier  (must be > 1)
        warmup_epoch:     number of *steps* (not epochs) to warm up over
        after_scheduler:  scheduler to use after warmup completes
    """
    def __init__(self, optimizer, multiplier, warmup_epoch, after_scheduler, last_epoch=-1):
        if multiplier <= 1.0:
            raise ValueError('multiplier must be > 1')
        self.multiplier      = multiplier
        self.warmup_epoch    = warmup_epoch
        self.after_scheduler = after_scheduler
        self.finished        = False
        super().__init__(optimizer, last_epoch=last_epoch)

    def get_lr(self):
        if self.last_epoch > self.warmup_epoch:
            return self.after_scheduler.get_lr()
        return [base_lr / self.multiplier *
                ((self.multiplier - 1.) * self.last_epoch / self.warmup_epoch + 1.)
                for base_lr in self.base_lrs]

    def step(self, epoch=None):
        epoch = (self.last_epoch + 1) if epoch is None else epoch
        self.last_epoch = epoch
        if epoch > self.warmup_epoch:
            self.after_scheduler.step(epoch - self.warmup_epoch)
        else:
            super().step(epoch)

    def state_dict(self):
        state = {k: v for k, v in self.__dict__.items()
                 if k not in ('optimizer', 'after_scheduler')}
        state['after_scheduler'] = self.after_scheduler.state_dict()
        return state

    def load_state_dict(self, state_dict):
        after_sd = state_dict.pop('after_scheduler')
        self.__dict__.update(state_dict)
        self.after_scheduler.load_state_dict(after_sd)


def get_scheduler(optimizer, n_iter_per_epoch,
                  lr_scheduler='cosine',
                  epochs=30,
                  warmup_epoch=5,
                  warmup_multiplier=100,
                  lr_decay_rate=0.1,
                  lr_steps=None,
                  eta_min=1e-5):
    """
    Build the full LR schedule.

    Args:
        optimizer:         the optimizer to wrap
        n_iter_per_epoch:  number of batches per epoch (len(train_loader))
        lr_scheduler:      'cosine' or 'step'
        epochs:            total training epochs
        warmup_epoch:      epochs for linear warmup (0 = no warmup)
        warmup_multiplier: LR starts at base_lr / warmup_multiplier
        lr_decay_rate:     gamma for MultiStepLR
        lr_steps:          epoch milestones for MultiStepLR (e.g. [20, 25])
        eta_min:           minimum LR for CosineAnnealingLR
    """
    if 'cosine' in lr_scheduler:
        after_scheduler = CosineAnnealingLR(
            optimizer=optimizer,
            eta_min=eta_min,
            T_max=(epochs - warmup_epoch) * n_iter_per_epoch)
    elif 'step' in lr_scheduler:
        milestones = [(m - warmup_epoch) * n_iter_per_epoch
                      for m in (lr_steps or [20, 25])]
        after_scheduler = MultiStepLR(
            optimizer=optimizer,
            gamma=lr_decay_rate,
            milestones=milestones)
    else:
        raise NotImplementedError(f"Scheduler '{lr_scheduler}' not supported. Use 'cosine' or 'step'.")

    if warmup_epoch > 0:
        return GradualWarmupScheduler(
            optimizer,
            multiplier=warmup_multiplier,
            after_scheduler=after_scheduler,
            warmup_epoch=warmup_epoch * n_iter_per_epoch)
    return after_scheduler

## FBResNet (from `base_module.py`)

In [ ]:
# ════════════════════════════════════════════════════════════════════
# mSEModule — multi-scale temporal Squeeze-and-Excitation
# ════════════════════════════════════════════════════════════════════
class mSEModule(nn.Module):
    def __init__(self, channel, n_segment=8, index=1):
        super().__init__()
        self.channel = channel; self.reduction = 16; self.n_segment = n_segment
        r = self.reduction
        self.conv1 = nn.Conv2d(channel, channel//r, kernel_size=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channel//r)
        self.conv2 = nn.Conv2d(channel//r, channel//r, kernel_size=3, padding=1,
                               groups=channel//r, bias=False)
        self.avg_pool_forward2  = nn.AvgPool2d(kernel_size=2, stride=2)
        self.avg_pool_forward4  = nn.AvgPool2d(kernel_size=4, stride=4)
        self.sigmoid_forward    = nn.Sigmoid()
        self.avg_pool_backward2 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.avg_pool_backward4 = nn.AvgPool2d(kernel_size=4, stride=4)
        self.sigmoid_backward   = nn.Sigmoid()
        self.pad1_forward  = (0, 0, 0, 0, 0, 0, 0, 1)
        self.pad1_backward = (0, 0, 0, 0, 0, 0, 1, 0)
        self.conv3             = nn.Conv2d(channel//r, channel,   kernel_size=1, bias=False)
        self.bn3               = nn.BatchNorm2d(channel)
        self.conv3_smallscale2 = nn.Conv2d(channel//r, channel//r, kernel_size=3, padding=1, bias=False)
        self.bn3_smallscale2   = nn.BatchNorm2d(channel//r)
        self.conv3_smallscale4 = nn.Conv2d(channel//r, channel//r, kernel_size=3, padding=1, bias=False)
        self.bn3_smallscale4   = nn.BatchNorm2d(channel//r)

    def forward(self, x):
        bottleneck = self.bn1(self.conv1(x))
        rb  = bottleneck.view((-1, self.n_segment) + bottleneck.size()[1:])
        t_fea_fwd, _   = rb.split([self.n_segment-1, 1], dim=1)
        _, t_fea_bwd   = rb.split([1, self.n_segment-1], dim=1)
        cb  = self.conv2(bottleneck)
        rcb = cb.view((-1, self.n_segment) + cb.size()[1:])
        _, tP1_fwd   = rcb.split([1, self.n_segment-1], dim=1)
        tP1_bwd, _   = rcb.split([self.n_segment-1, 1], dim=1)
        diff_fwd = tP1_fwd - t_fea_fwd;  diff_bwd = tP1_bwd - t_fea_bwd
        dfz = F.pad(diff_fwd, self.pad1_forward,  mode="constant", value=0).view((-1,)+diff_fwd.size()[2:])
        dbz = F.pad(diff_bwd, self.pad1_backward, mode="constant", value=0).view((-1,)+diff_bwd.size()[2:])
        yf2 = self.bn3_smallscale2(self.conv3_smallscale2(self.avg_pool_forward2(dfz)))
        yb2 = self.bn3_smallscale2(self.conv3_smallscale2(self.avg_pool_backward2(dbz)))
        yf4 = self.bn3_smallscale4(self.conv3_smallscale4(dfz))
        yb4 = self.bn3_smallscale4(self.conv3_smallscale4(dbz))
        yf2 = F.interpolate(yf2, dfz.size()[2:]); yb2 = F.interpolate(yb2, dbz.size()[2:])
        yf  = self.bn3(self.conv3(1/3*dfz + 1/3*yf2 + 1/3*yf4))
        yb  = self.bn3(self.conv3(1/3*dbz + 1/3*yb2 + 1/3*yb4))
        y   = 0.5*(self.sigmoid_forward(yf)-0.5) + 0.5*(self.sigmoid_backward(yb)-0.5)
        return x + x * y


# ════════════════════════════════════════════════════════════════════
# ShiftModule — temporal channel shift (TSM-style)
# ════════════════════════════════════════════════════════════════════
class ShiftModule(nn.Module):
    def __init__(self, input_channels, n_segment=8, n_div=8, mode='shift'):
        super().__init__()
        self.n_segment = n_segment
        self.fold      = input_channels // n_div
        self.conv      = nn.Conv1d(n_div*self.fold, n_div*self.fold,
                                   kernel_size=3, padding=1,
                                   groups=n_div*self.fold, bias=False)
        self.conv.weight.requires_grad = True
        self.conv.weight.data.zero_()
        self.conv.weight.data[:self.fold,         0, 2] = 1   # shift left  (future)
        self.conv.weight.data[self.fold:2*self.fold, 0, 0] = 1 # shift right (past)
        if 2*self.fold < input_channels:
            self.conv.weight.data[2*self.fold:, 0, 1] = 1     # identity

    def forward(self, x):
        nt, c, h, w = x.size(); nb = nt // self.n_segment
        x = x.view(nb, self.n_segment, c, h, w).permute(0,3,4,2,1).contiguous()
        x = self.conv(x.view(nb*h*w, c, self.n_segment))
        x = x.view(nb, h, w, c, self.n_segment).permute(0,4,3,1,2).contiguous()
        return x.view(nt, c, h, w)


# ════════════════════════════════════════════════════════════════════
# Residual blocks
# ════════════════════════════════════════════════════════════════════
class Bottleneck(nn.Module):
    expansion = 4
    def __init__(self, num_segments, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(inplanes, planes,   1, bias=True); self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes,   planes,   3, stride=stride, padding=1, bias=True)
        self.bn2   = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes,   planes*4, 1, bias=True); self.bn3 = nn.BatchNorm2d(planes*4)
        self.relu  = nn.ReLU(inplace=True); self.downsample = downsample
    def forward(self, x):
        r = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        return self.relu(out + (self.downsample(x) if self.downsample else r))


class BottleneckShift(nn.Module):
    """Bottleneck + mSEModule + ShiftModule — layers 2, 3, 4."""
    expansion = 4
    def __init__(self, num_segments, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(inplanes, planes,   1, bias=True); self.bn1 = nn.BatchNorm2d(planes)
        self.mse   = mSEModule(planes, n_segment=num_segments, index=1)
        self.shift = ShiftModule(planes, n_segment=num_segments, n_div=8, mode='shift')
        self.conv2 = nn.Conv2d(planes,   planes,   3, stride=stride, padding=1, bias=True)
        self.bn2   = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes,   planes*4, 1, bias=True); self.bn3 = nn.BatchNorm2d(planes*4)
        self.relu  = nn.ReLU(inplace=True); self.downsample = downsample
    def forward(self, x):
        r   = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.mse(out)
        out = self.shift(out)
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        return self.relu(out + (self.downsample(x) if self.downsample else r))


# ════════════════════════════════════════════════════════════════════
# FBResNet
# ════════════════════════════════════════════════════════════════════
class FBResNet(nn.Module):
    def __init__(self, num_segments, block, layers, num_classes=1000):
        super().__init__()
        self.inplanes = 64; self.num_segments = num_segments
        self.conv1   = nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=True)
        self.bn1     = nn.BatchNorm2d(64)
        self.relu    = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, stride=2, padding=1)
        # layer1 intentionally uses plain Bottleneck (matches original base_module.py)
        self.layer1  = self._make_layer(num_segments, Bottleneck,    64, layers[0])
        self.layer2  = self._make_layer(num_segments, block,        128, layers[1], stride=2)
        self.layer3  = self._make_layer(num_segments, block,        256, layers[2], stride=2)
        self.layer4  = self._make_layer(num_segments, block,        512, layers[3], stride=2)
        self.last_linear = nn.Linear(512 * block.expansion, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1); m.bias.data.zero_()

    def _make_layer(self, num_segments, block, planes, blocks, stride=1):
        ds = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            ds = nn.Sequential(
                nn.Conv2d(self.inplanes, planes*block.expansion, 1, stride=stride, bias=True),
                nn.BatchNorm2d(planes*block.expansion))
        layers = [block(num_segments, self.inplanes, planes, stride, ds)]
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(num_segments, self.inplanes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x))); x = self.maxpool(x)
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        return F.avg_pool2d(x, x.shape[2]).view(x.size(0), -1)


def fbresnet50(num_segments=8):
    return FBResNet(num_segments, BottleneckShift, [3, 4, 6, 3])

## TDN Model — `TDN_Net` + `TDN_TSN`

In [ ]:
class TDN_Net(nn.Module):
    """
    Faithful port of tdn_net.py using FBResNet backbone.
    Input:  (B*S, 15, H, W)
    Output: (B*S, num_classes)
    """
    def __init__(self, num_classes, num_segments=8):
        super().__init__()
        self.apha  = 0.5  if num_segments == 8 else 0.75
        self.belta = 0.5  if num_segments == 8 else 0.25

        r_main = fbresnet50(num_segments)
        r_diff = fbresnet50(num_segments)

        # RGB stream stem
        self.conv1   = r_main.conv1
        self.bn1     = r_main.bn1
        self.relu    = nn.ReLU(inplace=True)
        self.maxpool = r_main.maxpool

        # Diff stream: conv1_5 — 12-channel inflated from r_diff's first conv
        w_rgb  = r_diff.conv1.weight.data                              # (64, 3, 7, 7)
        w_12ch = w_rgb.mean(dim=1, keepdim=True).expand(-1, 12, -1, -1).contiguous()
        self.conv1_5 = nn.Sequential(
            nn.Conv2d(12, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.conv1_5[0].weight.data = w_12ch

        self.avg_diff     = nn.AvgPool2d(kernel_size=2, stride=2)
        self.maxpool_diff = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.diff_layer1  = r_diff.layer1   # plain Bottleneck, no shift

        self.layer1  = r_main.layer1        # plain Bottleneck
        self.layer2  = r_main.layer2        # BottleneckShift (mSE + Shift)
        self.layer3  = r_main.layer3        # BottleneckShift
        self.layer4  = r_main.layer4        # BottleneckShift
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc      = nn.Linear(r_main.last_linear.in_features, num_classes)
        nn.init.normal_(self.fc.weight, 0, 0.001)
        nn.init.constant_(self.fc.bias, 0)

    def forward(self, x):
        x1,x2,x3,x4,x5 = x[:,0:3],x[:,3:6],x[:,6:9],x[:,9:12],x[:,12:15]

        # Diff stream
        diffs       = torch.cat([x2-x1, x3-x2, x4-x3, x5-x4], dim=1)  # (B*S, 12, H, W)
        x_c5        = self.conv1_5(self.avg_diff(diffs))                # (B*S, 64, H/4, W/4)
        x_diff_stem = self.maxpool_diff(x_c5)                          # (B*S, 64, H/8, W/8)
        x_diff_l1   = self.diff_layer1(x_diff_stem)                    # (B*S, 256, H/8, W/8)

        # RGB stream
        x = self.relu(self.bn1(self.conv1(x3)))   # (B*S, 64, H/2, W/2)
        x = self.maxpool(x)                       # (B*S, 64, H/4, W/4)

        # Fusion 1 (before layer1)
        x = self.apha*x + self.belta*F.interpolate(x_diff_stem, x.shape[2:], mode='nearest')
        x = self.layer1(x)                        # (B*S, 256, H/8, W/8)

        # Fusion 2 (after layer1)
        x = self.apha*x + self.belta*F.interpolate(x_diff_l1, x.shape[2:], mode='nearest')

        x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        return self.fc(self.avgpool(x).flatten(1))


class TDN_TSN(nn.Module):
    """
    TSN wrapper: (B, S, 15, H, W) → TDN_Net → consensus → (B, num_classes).
    """
    def __init__(self, num_classes, num_segments=8, freeze=False):
        super().__init__()
        self.num_segments = num_segments
        self.backbone     = TDN_Net(num_classes, num_segments)
        if freeze:
            frozen = ('backbone.conv1','backbone.bn1',
                      'backbone.layer1','backbone.layer2',
                      'backbone.layer3','backbone.layer4')
            for n, p in self.named_parameters():
                if any(n.startswith(f) for f in frozen):
                    p.requires_grad = False
            print("Backbone frozen — conv1_5 / diff_layer1 / fc remain trainable.")

    def forward(self, x):
        B, S = x.shape[:2]
        logits = self.backbone(x.view(B*S, *x.shape[2:]))  # (B*S, C)
        return logits.view(B, S, -1).mean(1)               # (B, C)


def setup_model(num_classes, num_segments=8, freeze=False):
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    model     = TDN_TSN(num_classes, num_segments, freeze).to(device)
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parameters: {total:,} total | {trainable:,} trainable")
    return model, device

## Data Utilities

In [ ]:
def download_data():
    local = "./kth_dataset"
    if os.path.exists(local): print(f"Dataset at {local}"); return local
    safe = "/cephyr/users/domingos/Alvis/data_cache"
    os.makedirs(safe, exist_ok=True); os.environ["KAGGLEHUB_CACHE_DIR"] = safe
    p = kagglehub.dataset_download("vafaeii/kth-action-recognition-dataset")
    print("Downloaded to:", p); return p

def visualize_data(dataset_path):
    plt.figure(figsize=(30, 30))
    classes = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]
    for counter, idx in enumerate(random.choices(range(len(classes)), k=20), 1):
        cls  = classes[idx]
        vids = os.listdir(os.path.join(dataset_path, cls))
        cap  = cv2.VideoCapture(os.path.join(dataset_path, cls, random.choice(vids)), cv2.CAP_FFMPEG)
        ok, f = cap.read(); cap.release()
        if not ok: continue
        f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        cv2.putText(f, cls, (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,200,0), 1)
        plt.subplot(5,4,counter); plt.imshow(f); plt.axis('off')
    plt.tight_layout(); plt.show()

def partition_kth_dataset(orig_dir, target_dir="./kth_split",
                           test_split=0.2, val_split=0.2, seed=0):
    if os.path.exists(os.path.join(target_dir, "train")):
        print(f"Already partitioned at '{target_dir}'."); return target_dir
    random.seed(seed)
    subjects = [f"{i:02d}" for i in range(1,26)]; random.shuffle(subjects)
    nt = int(len(subjects)*test_split); nv = int((len(subjects)-nt)*val_split)
    test_p, val_p, train_p = subjects[:nt], subjects[nt:nt+nv], subjects[nt+nv:]
    print(f"Splits → Train:{len(train_p)} Val:{len(val_p)} Test:{len(test_p)}")
    os.makedirs(target_dir, exist_ok=True)
    with open(os.path.join(target_dir,"split_records.json"),"w") as f:
        json.dump({"train":train_p,"validation":val_p,"test":test_p}, f, indent=4)
    for line in open(os.path.join(orig_dir,"00sequences.txt")):
        line = line.strip()
        if not line or "frames" not in line or "*missing*" in line: continue
        parts=line.split(); vid=parts[0]
        ranges=re.findall(r'(\d+)-(\d+)'," ".join(parts[2:]))
        pid=vid.split('_')[0].replace('person',''); action=vid.split('_')[1]
        split = 'train' if pid in train_p else 'validation' if pid in val_p else 'test' if pid in test_p else None
        if split is None: continue
        src = os.path.join(orig_dir, action, f"{vid}_uncomp.avi")
        if not os.path.exists(src): continue
        out_dir = os.path.join(target_dir, split, action); os.makedirs(out_dir, exist_ok=True)
        cap = cv2.VideoCapture(src)
        fps,w,h = cap.get(cv2.CAP_PROP_FPS) or 25., int(cap.get(3)), int(cap.get(4))
        fourcc  = cv2.VideoWriter_fourcc(*'XVID')
        for i,(s,e) in enumerate(ranges):
            op = os.path.join(out_dir, f"{vid}_chunk{i}.avi")
            if os.path.exists(op): continue
            ow = cv2.VideoWriter(op, fourcc, fps, (w,h))
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(s)-1)
            for _ in range(int(s),int(e)+1):
                ret,fr = cap.read()
                if not ret: break
                ow.write(fr)
            ow.release()
        cap.release()
    print("Partitioning complete!"); return target_dir

def _find_classes(d):
    classes = sorted([x.name for x in os.scandir(d) if x.is_dir()])
    return classes, {c:i for i,c in enumerate(classes)}

def get_samples(root, extensions=(".mp4",".avi")):
    _, cls_idx = _find_classes(root)
    return make_dataset(root, cls_idx, extensions=extensions)

## TDN Dataset — `KTHTDNDataset` (with group transforms)

### What changed from v2
The dataset now reads frames as **PIL Images** and passes the 5-frame list through  
the group transform pipeline, ensuring all 5 frames share one crop window and  
one flip decision. The output per segment is still `(15, H, W)`.

```
5 PIL frames → GroupMultiScaleCrop (shared crop) → GroupRandomHorizontalFlip (shared flip)
             → Stack()     → (15, H, W) float tensor in [0, 1]
             → GroupNormalize → (15, H, W) normalised tensor
```


In [ ]:
class KTHTDNDataset(torch.utils.data.Dataset):
    """
    Returns (num_segments, 15, H, W) per video.

    Frames are read as PIL Images and passed through group transforms,
    guaranteeing crop/flip consistency across all 5 frames of each segment.

    train=True  → GroupMultiScaleCrop + GroupRandomHorizontalFlip (random each segment)
    train=False → GroupScale(256) + GroupCenterCrop(224) (deterministic)
    """
    def __init__(self, root, num_segments=8, train=True):
        super().__init__()
        self.num_segments  = num_segments
        self.train         = train
        self.transform     = make_train_transform() if train else make_eval_transform()
        min_frames = num_segments * 5
        self.samples = []
        for path, target in get_samples(root):
            cap = cv2.VideoCapture(path)
            n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
            if n >= min_frames: self.samples.append((path, target, n))
            else: print(f"Skip ({n} frames): {path}")

    def __len__(self): return len(self.samples)

    def _read_pil(self, cap, idx, total):
        """Read frame `idx` (clamped) and return a PIL RGB Image."""
        idx = max(0, min(idx, total-1))
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, bgr = cap.read()
        if not ret: cap.set(cv2.CAP_PROP_POS_FRAMES, 0); _, bgr = cap.read()
        return Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))

    def _anchor_indices(self, total):
        bin_size = total / self.num_segments
        anchors  = []
        for s in range(self.num_segments):
            lo = int(s * bin_size)
            hi = min(int((s+1)*bin_size)-1, total-1)
            anchors.append(random.randint(lo, hi) if self.train else (lo+hi)//2)
        return anchors

    def __getitem__(self, idx):
        path, target, total = self.samples[idx]
        cap     = cv2.VideoCapture(path)
        anchors = self._anchor_indices(total)

        segments = []
        for t in anchors:
            # 5 PIL frames: [t-2, t-1, t, t+1, t+2], clamped
            pil_frames = [self._read_pil(cap, t+off, total) for off in (-2,-1,0,1,2)]
            # Group transforms: shared crop + flip → Stack → Normalize
            # apply_group_transform returns (15, H, W) tensor
            seg_tensor = apply_group_transform(pil_frames, self.transform)
            segments.append(seg_tensor)

        cap.release()
        return torch.stack(segments, dim=0), target  # (S, 15, H, W), int

## Training Function

### Changes from v2
- Uses `get_scheduler` (warmup + cosine/step) instead of fixed-LR Adam
- Calls `scheduler.step()` after every batch (as in the original `main.py`)
- `clip_grad_norm_` added (the original clips at `20.0`)
- Partial BN: after unfreeze, BN layers beyond the first are kept frozen  
  (`partialBN` behaviour from `models.py`) — controlled by `PARTIAL_BN`


In [ ]:
def train_model(model, dataloaders, device, class_names, save_dir,
                num_epochs=30, base_lr=1e-3, optimizer_type='adam',
                lr_scheduler_type='cosine', warmup_epoch=5,
                warmup_multiplier=100, lr_steps=None,
                freeze=False, unfreeze_epoch=None,
                clip_gradient=20.0, partial_bn=True):

    num_classes = len(class_names)
    acc_metric  = MulticlassAccuracy(num_classes=num_classes, average='micro').to(device)
    criterion   = nn.CrossEntropyLoss()

    def make_optimizer(params):
        if optimizer_type == 'sgd':
            return optim.SGD(params, lr=base_lr, momentum=0.9, weight_decay=1e-4)
        return optim.Adam(params, lr=base_lr, weight_decay=1e-4)

    trainable = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = make_optimizer(trainable)

    n_iter = len(dataloaders['train'])
    scheduler = get_scheduler(optimizer, n_iter,
                               lr_scheduler=lr_scheduler_type,
                               epochs=num_epochs,
                               warmup_epoch=warmup_epoch if warmup_epoch < num_epochs else 0,
                               warmup_multiplier=warmup_multiplier,
                               lr_steps=lr_steps or [int(num_epochs*0.67), int(num_epochs*0.83)])

    history   = {k:[] for k in ('train_loss','train_acc','val_loss','val_acc')}
    best_acc  = 0.0
    best_path = os.path.join(save_dir, "best_model.pth")

    print("Starting Training...")
    for epoch in range(num_epochs):

        # Full-backbone unfreeze
        if freeze and unfreeze_epoch and epoch == unfreeze_epoch:
            print("Unfreezing all parameters...")
            for p in model.parameters(): p.requires_grad = True
            optimizer = make_optimizer(model.parameters())
            scheduler = get_scheduler(optimizer, n_iter,
                                       lr_scheduler=lr_scheduler_type,
                                       epochs=num_epochs - epoch,
                                       warmup_epoch=0,
                                       lr_steps=[int((num_epochs-epoch)*0.67)])

        print(f"\nEpoch {epoch+1}/{num_epochs}  (LR: {optimizer.param_groups[0]['lr']:.2e})")
        print("-" * 20)

        for phase in ('train', 'validation'):
            if phase == 'train':
                model.train()
                # Partial BN: freeze all BN except the first one after unfreeze
                if partial_bn:
                    bn_count = 0
                    for m in model.modules():
                        if isinstance(m, nn.BatchNorm2d):
                            bn_count += 1
                            if bn_count >= 2:
                                m.eval()
                                m.weight.requires_grad = False
                                m.bias.requires_grad   = False
            else:
                model.eval()

            running_loss = 0.0; n_batches = 0
            for clips, labels in dataloaders[phase]:
                clips, labels = clips.to(device), labels.to(device)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    logits = model(clips)
                    loss   = criterion(logits, labels)
                    if phase == 'train':
                        loss.backward()
                        if clip_gradient:
                            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_gradient)
                        optimizer.step()
                        scheduler.step()
                _, preds = torch.max(logits, 1)
                running_loss += loss.item(); n_batches += 1
                acc_metric.update(preds, labels)

            epoch_loss = running_loss / n_batches
            epoch_acc  = acc_metric.compute().item(); acc_metric.reset()
            print(f"{phase.capitalize()}: loss={epoch_loss:.4f}  acc={epoch_acc*100:.2f}%")

            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc)
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc)
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    torch.save({'epoch': epoch, 'state_dict': model.state_dict(),
                                'optimizer': optimizer.state_dict(),
                                'scheduler': scheduler.state_dict(),
                                'best_acc': best_acc}, best_path)
                    print(f"  → Best saved ({best_acc*100:.2f}%)")

    print(f"\nRestoring best weights ({best_acc*100:.2f}%)")
    model.load_state_dict(torch.load(best_path)['state_dict'])
    return model, history

## Evaluation Function

In [ ]:
def evaluate_model(model, dataloader, device, class_names, save_dir):
    model.eval()
    nc  = len(class_names)
    am  = MulticlassAccuracy(num_classes=nc).to(device)
    cm  = MulticlassConfusionMatrix(num_classes=nc).to(device)
    all_p, all_l = [], []
    with torch.no_grad():
        for clips, labels in dataloader:
            clips, labels = clips.to(device), labels.to(device)
            logits = model(clips); _, preds = torch.max(logits, 1)
            am.update(preds, labels); cm.update(preds, labels)
            all_p.extend(preds.cpu().numpy()); all_l.extend(labels.cpu().numpy())
    acc = am.compute().item()
    print(f"\nTest Accuracy: {acc*100:.2f}%")
    report = classification_report(all_l, all_p, target_names=class_names)
    print(report)
    with open(os.path.join(save_dir, "classification_report.txt"), "w") as f:
        f.write(f"Accuracy: {acc*100:.2f}%\n\n"); f.write(report)
    plt.figure(figsize=(10,8))
    sns.heatmap(cm.compute().cpu().numpy(), annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix — TDN')
    plt.savefig(os.path.join(save_dir,"confusion_matrix.png"), dpi=300, bbox_inches='tight')
    plt.show()

## Visualizing Predictions

In [ ]:
def visualize_predictions(model, dataset, device, class_names, save_dir, num_videos=4):
    model.eval()
    indices = random.sample(range(len(dataset)), num_videos)
    clips   = torch.stack([dataset[i][0] for i in indices])
    labels  = torch.tensor([dataset[i][1] for i in indices])
    with torch.no_grad():
        _, preds = torch.max(model(clips.to(device)), 1)
    preds = preds.cpu()
    # De-normalise for display: GroupNormalize uses ImageNet stats repeated across 15 ch
    mean = torch.tensor([0.485,0.456,0.406]).repeat(5).view(15,1,1)
    std  = torch.tensor([0.229,0.224,0.225]).repeat(5).view(15,1,1)
    fig, axes = plt.subplots(1, num_videos, figsize=(16,4))
    if num_videos == 1: axes = [axes]
    for i in range(num_videos):
        anchor = clips[i, clips.shape[1]//2, 6:9]   # middle segment, anchor frame (ch 6-8)
        anchor = torch.clamp(anchor * std[6:9] + mean[6:9], 0, 1)
        color  = "green" if labels[i]==preds[i] else "red"
        axes[i].imshow(anchor.permute(1,2,0).numpy())
        axes[i].set_title(f"True: {class_names[labels[i]]}\nPred: {class_names[preds[i]]}",
                          color=color, fontweight='bold')
        axes[i].axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir,"predictions.png"), dpi=300, bbox_inches='tight')
    plt.show()

## Learning Curves

In [ ]:
def plot_learning_curves(history, save_dir):
    ep = range(1, len(history['train_loss'])+1)
    fig, (ax1,ax2) = plt.subplots(1,2,figsize=(14,5))
    ax1.plot(ep,history['train_loss'],'b-o',label='Train')
    ax1.plot(ep,history['val_loss'],  'r-o',label='Val')
    ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.5)
    ax2.plot(ep,history['train_acc'], 'b-o',label='Train')
    ax2.plot(ep,history['val_acc'],   'r-o',label='Val')
    ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir,"learning_curves.png"), dpi=300, bbox_inches='tight')
    plt.show()

## Main Execution

### Hyperparameter guide

| Parameter | Recommended (fine-tune) | Recommended (full train) |
|---|---|---|
| `OPTIMIZER` | `'adam'` | `'sgd'` |
| `LR_SCHEDULER` | `'cosine'` | `'step'` |
| `BASE_LR` | `1e-3` | `1e-2` |
| `WARMUP_EPOCH` | `3` | `5` |
| `PARTIAL_BN` | `True` | `True` |
| `CLIP_GRADIENT` | `20.0` | `20.0` |

### Memory guidance
Each batch item is `(S, 15, H, W)`. For `S=8` and `H=W=224`:  
`4 × 8 × 15 × 224 × 224 × 4 bytes ≈ 483 MB` of input alone.  
Start with `BATCH_SIZE=4`; reduce to `2` or `1` if OOM.


In [ ]:
# ── Hyperparameters ─────────────────────────────────────────────────────────
SEED              = 0
TEST_SPLIT        = 0.2
VAL_SPLIT         = 0.2
BATCH_SIZE        = 4
NR_EPOCHS         = 30
BASE_LR           = 1e-3
OPTIMIZER         = 'adam'          # 'adam' or 'sgd'
LR_SCHEDULER      = 'cosine'        # 'cosine' or 'step'
LR_STEPS          = None            # e.g. [20, 25] for 'step' scheduler
WARMUP_EPOCH      = 3               # 0 = no warmup
WARMUP_MULTIPLIER = 100
NUM_SEGMENTS      = 8               # 8 or 16
FREEZE            = True
UNFREEZE_EPOCH    = 15
CLIP_GRADIENT     = 20.0
PARTIAL_BN        = True

CLASS_NAMES = ['boxing','handclapping','handwaving','jogging','running','walking']

dataloader_kwargs = {
    'num_workers': 8, 'pin_memory': True,
    'persistent_workers': True, 'prefetch_factor': 2,
}

# ── Setup ────────────────────────────────────────────────────────────────────
set_seed(SEED)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir   = f"./runs/tdn_v3_{timestamp}"
os.makedirs(run_dir, exist_ok=True)
print(f"Run directory: {run_dir}")

with open(os.path.join(run_dir, "hyperparams.json"), "w") as f:
    json.dump({"SEED":SEED, "BATCH_SIZE":BATCH_SIZE, "NR_EPOCHS":NR_EPOCHS,
               "BASE_LR":BASE_LR, "OPTIMIZER":OPTIMIZER,
               "LR_SCHEDULER":LR_SCHEDULER, "WARMUP_EPOCH":WARMUP_EPOCH,
               "NUM_SEGMENTS":NUM_SEGMENTS, "FREEZE":FREEZE,
               "UNFREEZE_EPOCH":UNFREEZE_EPOCH, "PARTIAL_BN":PARTIAL_BN,
               "CLIP_GRADIENT":CLIP_GRADIENT,
               "MODEL":"TDN FBResNet-50 (mSE+Shift), group transforms"}, f, indent=4)

# ── Data ────────────────────────────────────────────────────────────────────
data_path = download_data()
visualize_data(data_path)
split_dir = partition_kth_dataset(data_path, "./kth_split",
                                   TEST_SPLIT, VAL_SPLIT, SEED)

train_dir, val_dir, test_dir = [os.path.join(split_dir, s)
                                 for s in ('train','validation','test')]

# ── Datasets — no transform arg: group transforms are built inside the class ─
dataset_train = KTHTDNDataset(train_dir, num_segments=NUM_SEGMENTS, train=True)
dataset_val   = KTHTDNDataset(val_dir,   num_segments=NUM_SEGMENTS, train=False)

dataloaders = {
    'train':      DataLoader(dataset_train, batch_size=BATCH_SIZE,
                             shuffle=True,  **dataloader_kwargs),
    'validation': DataLoader(dataset_val,   batch_size=BATCH_SIZE,
                             shuffle=False, **dataloader_kwargs),
}

# ── Model ────────────────────────────────────────────────────────────────────
print("\nBuilding TDN (FBResNet-50 + mSE + Shift + group transforms + warmup scheduler)...")
model, device = setup_model(len(CLASS_NAMES), num_segments=NUM_SEGMENTS, freeze=FREEZE)

# ── Train ────────────────────────────────────────────────────────────────────
model, history = train_model(
    model=model, dataloaders=dataloaders, device=device,
    class_names=CLASS_NAMES, save_dir=run_dir,
    num_epochs=NR_EPOCHS, base_lr=BASE_LR,
    optimizer_type=OPTIMIZER, lr_scheduler_type=LR_SCHEDULER,
    warmup_epoch=WARMUP_EPOCH, warmup_multiplier=WARMUP_MULTIPLIER,
    lr_steps=LR_STEPS, freeze=FREEZE, unfreeze_epoch=UNFREEZE_EPOCH,
    clip_gradient=CLIP_GRADIENT, partial_bn=PARTIAL_BN)

plot_learning_curves(history, save_dir=run_dir)

# ── Evaluate ─────────────────────────────────────────────────────────────────
print("\nFinal evaluation on test set...")
dataset_test    = KTHTDNDataset(test_dir, num_segments=NUM_SEGMENTS, train=False)
dataloader_test = DataLoader(dataset_test, batch_size=BATCH_SIZE, **dataloader_kwargs)

evaluate_model(model, dataloader_test, device, CLASS_NAMES, save_dir=run_dir)
visualize_predictions(model, dataset_test, device, CLASS_NAMES, save_dir=run_dir)